## IMPORT LIBRARY

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import math

## LOAD DATA

In [ ]:
df = pd.read_csv('decision.csv')

print("DATA:")
print(df.to_string(index=False))
print(f"\nTotal data: {len(df)}")

DATA:
Outlook Temperature Humidity Windy     Play
  Sunny         Hot     High    No DontPlay
  Sunny         Hot     High   Yes DontPlay
 Cloudy         Hot     High    No     Play
  Rainy        Mild     High    No     Play
  Rainy        Cold   Normal    No     Play
  Rainy        Cold   Normal   Yes     Play
 Cloudy        Cold   Normal   Yes     Play
  Sunny        Mild     High    No DontPlay
  Sunny        Cold   Normal    No     Play
  Sunny        Mild   Normal    No     Play
  Sunny        Mild   Normal   Yes     Play
  Sunny        Mild     High   Yes     Play
  Sunny         Hot   Normal    No     Play
  Rainy        Mild     High   Yes DontPlay
 Cloudy        Mild     High   Yes     Play
 Cloudy         Hot   Normal    No     Play
  Rainy        Mild     High   Yes DontPlay

Total data: 17


## FUNGSI DASAR

In [ ]:
def entropy(data):
    """Hitung entropy"""
    if len(data) == 0:
        return 0
    counts = Counter(data)
    total = len(data)
    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)
    return ent

def info_gain(df, attr, target):
    """Hitung information gain"""
    total_entropy = entropy(df[target])
    weighted_entropy = 0
    total = len(df)
    for value in df[attr].unique():
        subset = df[df[attr] == value]
        subset_entropy = entropy(subset[target])
        weight = len(subset) / total
        weighted_entropy += weight * subset_entropy
    return total_entropy - weighted_entropy

def split_info(df, attr):
    """Hitung split info"""
    total = len(df)
    split = 0
    for value in df[attr].unique():
        subset_size = len(df[df[attr] == value])
        if subset_size > 0:
            p = subset_size / total
            split -= p * math.log2(p)
    return split

def gain_ratio(df, attr, target):
    """Hitung gain ratio"""
    gain = info_gain(df, attr, target)
    split = split_info(df, attr)
    return gain / split if split != 0 else 0

## 1. Menentukan Root Node

In [ ]:
target = 'Play'
attributes = ['Outlook', 'Temperature', 'Humidity', 'Windy']

print("1. MENENTUKAN ROOT NODE")
print("=" * 60)

total_entropy = entropy(df[target])
print(f"Entropy total dataset: {total_entropy:.4f}\n")

results_level1 = {}

for attr in attributes:
    gain = info_gain(df, attr, target)
    split = split_info(df, attr)
    gr = gain_ratio(df, attr, target)
    results_level1[attr] = {'gain': gain, 'split': split, 'gain_ratio': gr}

    print(f"\n{attr}:")
    print(f"  Gain = {gain:.4f}")
    print(f"  Split Info = {split:.4f}")
    print(f"  Gain Ratio = {gr:.4f}")

# Tentukan root node
root_node = max(results_level1, key=lambda x: results_level1[x]['gain_ratio'])
print(f"\n{'='*60}")
print(f"✓ ROOT NODE: {root_node}")
print(f"  Gain Ratio = {results_level1[root_node]['gain_ratio']:.4f}")

1. MENENTUKAN ROOT NODE
Entropy total dataset: 0.8740


Outlook:
  Gain = 0.1393
  Split Info = 1.5222
  Gain Ratio = 0.0915

Temperature:
  Gain = 0.1393
  Split Info = 1.5222
  Gain Ratio = 0.0915

Humidity:
  Gain = 0.3493
  Split Info = 0.9975
  Gain Ratio = 0.3502

Windy:
  Gain = 0.0203
  Split Info = 0.9975
  Gain Ratio = 0.0203

✓ ROOT NODE: Humidity
  Gain Ratio = 0.3502


## 2. Split Berdasarkan Humidity

In [ ]:
root_node = 'Humidity'

print(f"2. SPLIT BERDASARKAN {root_node}")
print("=" * 60)

# Bagi data berdasarkan nilai Humidity
humidity_branches = {}
for value in df[root_node].unique():
    subset = df[df[root_node] == value]
    humidity_branches[value] = subset

    print(f"\n{root_node} = {value}:")
    print(f"  Jumlah data: {len(subset)}")
    print(f"  Data lengkap:")
    print(subset.to_string(index=False))
    print(f"  Distribusi Play: {dict(Counter(subset['Play']))}")

    branch_entropy = entropy(subset['Play'])
    print(f"  Entropy = {branch_entropy:.4f}")

    if len(Counter(subset['Play'])) == 1:
        print(f"  → LEAF: {subset['Play'].iloc[0]}")
    else:
        print(f"  → BUTUH SPLIT LAGI")

2. SPLIT BERDASARKAN Humidity

Humidity = High:
  Jumlah data: 9
  Data lengkap:
Outlook Temperature Humidity Windy     Play
  Sunny         Hot     High    No DontPlay
  Sunny         Hot     High   Yes DontPlay
 Cloudy         Hot     High    No     Play
  Rainy        Mild     High    No     Play
  Sunny        Mild     High    No DontPlay
  Sunny        Mild     High   Yes     Play
  Rainy        Mild     High   Yes DontPlay
 Cloudy        Mild     High   Yes     Play
  Rainy        Mild     High   Yes DontPlay
  Distribusi Play: {'DontPlay': 5, 'Play': 4}
  Entropy = 0.9911
  → BUTUH SPLIT LAGI

Humidity = Normal:
  Jumlah data: 8
  Data lengkap:
Outlook Temperature Humidity Windy Play
  Rainy        Cold   Normal    No Play
  Rainy        Cold   Normal   Yes Play
 Cloudy        Cold   Normal   Yes Play
  Sunny        Cold   Normal    No Play
  Sunny        Mild   Normal    No Play
  Sunny        Mild   Normal   Yes Play
  Sunny         Hot   Normal    No Play
 Cloudy         Hot 

## 3.1 Branch Humidity = Normal

In [ ]:
print("3.1 BRANCH Humidity = Normal")
print("=" * 60)

normal_data = humidity_branches['Normal']
print(f"Data Humidity = Normal:")
print(normal_data[['Humidity', 'Play']].to_string(index=False))
print(f"\nDistribusi: {dict(Counter(normal_data['Play']))}")

if len(Counter(normal_data['Play'])) == 1:
    print(f"\n✓ LEAF → Prediksi: {normal_data['Play'].iloc[0]}")
    print(f"  (Semua data dengan Humidity Normal menghasilkan Play)")

3.1 BRANCH Humidity = Normal
Data Humidity = Normal:
Humidity Play
  Normal Play
  Normal Play
  Normal Play
  Normal Play
  Normal Play
  Normal Play
  Normal Play
  Normal Play

Distribusi: {'Play': 8}

✓ LEAF → Prediksi: Play
  (Semua data dengan Humidity Normal menghasilkan Play)


## 3.2 Branch Humidity = High

In [ ]:
print("3.2 BRANCH Humidity = High")
print("=" * 60)

high_data = humidity_branches['High']
print("Data Humidity = High:")
print(high_data[['Outlook', 'Temperature', 'Windy', 'Play']].to_string(index=False))

print(f"\nDistribusi Play: {dict(Counter(high_data['Play']))}")
print(f"Entropy branch High = {entropy(high_data['Play']):.4f}")

# Atribut yang tersisa (selain Humidity)
remaining_attrs = ['Outlook', 'Temperature', 'Windy']

print("\nHitung gain ratio untuk atribut yang tersisa:\n")
high_results = {}

for attr in remaining_attrs:
    print(f"--- {attr} ---")

    # Hitung weighted entropy
    weighted_entropy = 0
    total = len(high_data)
    for value in high_data[attr].unique():
        subset = high_data[high_data[attr] == value]
        subset_entropy = entropy(subset['Play'])
        weight = len(subset) / total
        weighted_entropy += weight * subset_entropy
        print(f"  {attr}={value}: {len(subset)} data, entropy={subset_entropy:.4f}, weight={weight:.3f}")

    gain = entropy(high_data['Play']) - weighted_entropy

    # Split info
    split = 0
    for value in high_data[attr].unique():
        p = len(high_data[high_data[attr] == value]) / total
        if p > 0:
            split -= p * math.log2(p)

    gr = gain / split if split != 0 else 0
    high_results[attr] = {'gain': gain, 'split': split, 'gain_ratio': gr}
    print(f"  Gain = {gain:.4f}, Split Info = {split:.4f}, Gain Ratio = {gr:.4f}\n")

# Tentukan node untuk branch High
high_node = max(high_results, key=lambda x: high_results[x]['gain_ratio'])
print(f"✓ Untuk branch Humidity=High, node terbaik adalah: {high_node}")
print(f"  Gain Ratio = {high_results[high_node]['gain_ratio']:.4f}")

3.2 BRANCH Humidity = High
Data Humidity = High:
Outlook Temperature Windy     Play
  Sunny         Hot    No DontPlay
  Sunny         Hot   Yes DontPlay
 Cloudy         Hot    No     Play
  Rainy        Mild    No     Play
  Sunny        Mild    No DontPlay
  Sunny        Mild   Yes     Play
  Rainy        Mild   Yes DontPlay
 Cloudy        Mild   Yes     Play
  Rainy        Mild   Yes DontPlay

Distribusi Play: {'DontPlay': 5, 'Play': 4}
Entropy branch High = 0.9911

Hitung gain ratio untuk atribut yang tersisa:

--- Outlook ---
  Outlook=Sunny: 4 data, entropy=0.8113, weight=0.444
  Outlook=Cloudy: 2 data, entropy=0.0000, weight=0.222
  Outlook=Rainy: 3 data, entropy=0.9183, weight=0.333
  Gain = 0.3244, Split Info = 1.5305, Gain Ratio = 0.2120

--- Temperature ---
  Temperature=Hot: 3 data, entropy=0.9183, weight=0.333
  Temperature=Mild: 6 data, entropy=1.0000, weight=0.667
  Gain = 0.0183, Split Info = 0.9183, Gain Ratio = 0.0199

--- Windy ---
  Windy=No: 4 data, entropy=1.0000,

## 4. Split Humidity=High

In [ ]:
print("4. SPLIT Humidity=High BERDASARKAN Outlook")
print("=" * 60)

high_data = humidity_branches['High']

# Split berdasarkan Outlook
outlook_branches = {}
for outlook in high_data['Outlook'].unique():
    subset = high_data[high_data['Outlook'] == outlook]
    outlook_branches[outlook] = subset

    print(f"\nOutlook = {outlook}:")
    print(f"  Data: {subset[['Outlook', 'Windy', 'Play']].to_string(index=False)}")
    print(f"  Distribusi Play: {dict(Counter(subset['Play']))}")

    if len(Counter(subset['Play'])) == 1:
        print(f"  → LEAF: {subset['Play'].iloc[0]}")
    else:
        print(f"  → BUTUH SPLIT LAGI")

4. SPLIT Humidity=High BERDASARKAN Outlook

Outlook = Sunny:
  Data: Outlook Windy     Play
  Sunny    No DontPlay
  Sunny   Yes DontPlay
  Sunny    No DontPlay
  Sunny   Yes     Play
  Distribusi Play: {'DontPlay': 3, 'Play': 1}
  → BUTUH SPLIT LAGI

Outlook = Cloudy:
  Data: Outlook Windy Play
 Cloudy    No Play
 Cloudy   Yes Play
  Distribusi Play: {'Play': 2}
  → LEAF: Play

Outlook = Rainy:
  Data: Outlook Windy     Play
  Rainy    No     Play
  Rainy   Yes DontPlay
  Rainy   Yes DontPlay
  Distribusi Play: {'Play': 1, 'DontPlay': 2}
  → BUTUH SPLIT LAGI


## 5.1 Branch Outlook = Rainy

In [ ]:
print("5.1 BRANCH Outlook = Rainy (dalam Humidity=High)")
print("=" * 60)

rainy_data = outlook_branches['Rainy']
print("Data Outlook = Rainy:")
print(rainy_data[['Outlook', 'Windy', 'Play']].to_string(index=False))

print(f"\nDistribusi Play: {dict(Counter(rainy_data['Play']))}")

# Split berdasarkan Windy
print("\nSplit berdasarkan Windy:")
for windy in rainy_data['Windy'].unique():
    subset = rainy_data[rainy_data['Windy'] == windy]
    print(f"\n  Windy = {windy}:")
    print(f"    Data: {subset[['Windy', 'Play']].to_string(index=False)}")
    print(f"    Distribusi: {dict(Counter(subset['Play']))}")

    if len(Counter(subset['Play'])) == 1:
        print(f"    → LEAF: {subset['Play'].iloc[0]}")

5.1 BRANCH Outlook = Rainy (dalam Humidity=High)
Data Outlook = Rainy:
Outlook Windy     Play
  Rainy    No     Play
  Rainy   Yes DontPlay
  Rainy   Yes DontPlay

Distribusi Play: {'Play': 1, 'DontPlay': 2}

Split berdasarkan Windy:

  Windy = No:
    Data: Windy Play
   No Play
    Distribusi: {'Play': 1}
    → LEAF: Play

  Windy = Yes:
    Data: Windy     Play
  Yes DontPlay
  Yes DontPlay
    Distribusi: {'DontPlay': 2}
    → LEAF: DontPlay


## 5.2 Branch Outlook = Sunny

In [ ]:
print("5.2 BRANCH Outlook = Sunny (dalam Humidity=High)")
print("=" * 60)

sunny_data = outlook_branches['Sunny']
print("Data Outlook = Sunny:")
print(sunny_data[['Outlook', 'Temperature', 'Windy', 'Play']].to_string(index=False))

print(f"\nDistribusi Play: {dict(Counter(sunny_data['Play']))}")

# Split berdasarkan Windy
print("\nSplit berdasarkan Windy:")
sunny_windy_branches = {}
for windy in sunny_data['Windy'].unique():
    subset = sunny_data[sunny_data['Windy'] == windy]
    sunny_windy_branches[windy] = subset
    print(f"\n  Windy = {windy}:")
    print(f"    Data: {subset[['Windy', 'Temperature', 'Play']].to_string(index=False)}")
    print(f"    Distribusi: {dict(Counter(subset['Play']))}")

    if len(Counter(subset['Play'])) == 1:
        print(f"    → LEAF: {subset['Play'].iloc[0]}")
    else:
        print(f"    → BUTUH SPLIT LAGI dengan Temperature")

5.2 BRANCH Outlook = Sunny (dalam Humidity=High)
Data Outlook = Sunny:
Outlook Temperature Windy     Play
  Sunny         Hot    No DontPlay
  Sunny         Hot   Yes DontPlay
  Sunny        Mild    No DontPlay
  Sunny        Mild   Yes     Play

Distribusi Play: {'DontPlay': 3, 'Play': 1}

Split berdasarkan Windy:

  Windy = No:
    Data: Windy Temperature     Play
   No         Hot DontPlay
   No        Mild DontPlay
    Distribusi: {'DontPlay': 2}
    → LEAF: DontPlay

  Windy = Yes:
    Data: Windy Temperature     Play
  Yes         Hot DontPlay
  Yes        Mild     Play
    Distribusi: {'DontPlay': 1, 'Play': 1}
    → BUTUH SPLIT LAGI dengan Temperature


## 6. Branch Sunny, Windy=Yes

In [ ]:
print("6. BRANCH Sunny, Windy = Yes (dalam Humidity=High, Outlook=Sunny)")
print("=" * 60)

sunny_windy_yes = sunny_windy_branches['Yes']
print("Data Sunny & Windy = Yes:")
print(sunny_windy_yes[['Temperature', 'Play']].to_string(index=False))

print(f"\nDistribusi Play: {dict(Counter(sunny_windy_yes['Play']))}")

# Split berdasarkan Temperature
print("\nSplit berdasarkan Temperature:")
for temp in sunny_windy_yes['Temperature'].unique():
    subset = sunny_windy_yes[sunny_windy_yes['Temperature'] == temp]
    print(f"\n  Temperature = {temp}:")
    print(f"    Data: {subset[['Temperature', 'Play']].to_string(index=False)}")
    print(f"    Distribusi: {dict(Counter(subset['Play']))}")

    if len(Counter(subset['Play'])) == 1:
        print(f"    → LEAF: {subset['Play'].iloc[0]}")

6. BRANCH Sunny, Windy = Yes (dalam Humidity=High, Outlook=Sunny)
Data Sunny & Windy = Yes:
Temperature     Play
        Hot DontPlay
       Mild     Play

Distribusi Play: {'DontPlay': 1, 'Play': 1}

Split berdasarkan Temperature:

  Temperature = Hot:
    Data: Temperature     Play
        Hot DontPlay
    Distribusi: {'DontPlay': 1}
    → LEAF: DontPlay

  Temperature = Mild:
    Data: Temperature Play
       Mild Play
    Distribusi: {'Play': 1}
    → LEAF: Play


## Decision Tree

In [ ]:
print("DECISION TREE")
print("=" * 80)

print("""
                    ┌──────────────┐
                    │   Humidity   │
                    └──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │                             │
        [Normal]                        [High]
            │                             │
            ▼                             ▼
        ┌───────┐                   ┌──────────┐
        │ Play  │                   │ Outlook  │
        └───────┘                   └──────────┘
                                          │
                        ┌─────────────────┼─────────────────┐
                        │                 │                 │
                    [Cloudy]           [Rainy]           [Sunny]
                        │                 │                 │
                        ▼                 ▼                 ▼
                    ┌───────┐        ┌────────┐        ┌────────┐
                    │ Play  │        │ Windy  │        │ Windy  │
                    └───────┘        └────────┘        └────────┘
                                          │                 │
                                      ┌───┴───┐         ┌───┴───┐
                                      │       │         │       │
                                     No      Yes       No      Yes
                                      │       │         │       │
                                      ▼       ▼         ▼       ▼
                                   ┌────┐  ┌──────┐  ┌──────┐ ┌────────────┐
                                   │Play│  │Dont  │  │Dont  │ │Temperature│
                                   └────┘  │Play  │  │Play  │ └────────────┘
                                          └──────┘  └──────┘        │
                                                                 ┌───┴───┐
                                                                 │       │
                                                               [Hot]   [Mild]
                                                                 │       │
                                                                 ▼       ▼
                                                              ┌──────┐ ┌──────┐
                                                              │Dont  │ │ Play │
                                                              │Play  │ └──────┘
                                                              └──────┘
""")

print("\n" + "=" * 80)
print("RULE SETS:")
print("=" * 80)
print("1. IF Humidity = Normal THEN Play")
print("2. IF Humidity = High AND Outlook = Cloudy THEN Play")
print("3. IF Humidity = High AND Outlook = Rainy AND Windy = No THEN Play")
print("4. IF Humidity = High AND Outlook = Rainy AND Windy = Yes THEN Don't Play")
print("5. IF Humidity = High AND Outlook = Sunny AND Windy = No THEN Don't Play")
print("6. IF Humidity = High AND Outlook = Sunny AND Windy = Yes AND Temperature = Hot THEN Don't Play.")
print("7. IF Humidity = High AND Outlook = Sunny AND Windy = Yes AND Temperature = Mild THEN Play.")

DECISION TREE

                    ┌──────────────┐
                    │   Humidity   │
                    └──────────────┘
                           │
            ┌──────────────┴──────────────┐
            │                             │
        [Normal]                        [High]
            │                             │
            ▼                             ▼
        ┌───────┐                   ┌──────────┐
        │ Play  │                   │ Outlook  │
        └───────┘                   └──────────┘
                                          │
                        ┌─────────────────┼─────────────────┐
                        │                 │                 │
                    [Cloudy]           [Rainy]           [Sunny]
                        │                 │                 │
                        ▼                 ▼                 ▼
                    ┌───────┐        ┌────────┐        ┌────────┐
                    │ Play  │        │ Windy  │     

## Validasi Semua data

In [ ]:
def predict_manual_corrected(row):
    """Prediksi berdasarkan tree"""
    # Level 1: Cek Humidity
    if row['Humidity'] == 'Normal':
        return 'Play'

    # Level 2: Humidity = High, cek Outlook
    if row['Humidity'] == 'High':
        if row['Outlook'] == 'Cloudy':
            return 'Play'
        elif row['Outlook'] == 'Rainy':
            # Cek Windy
            if row['Windy'] == 'No':
                return 'Play'
            else:  # Windy = Yes
                return 'DontPlay'
        else:  # Outlook = Sunny
            # Cek Windy
            if row['Windy'] == 'No':
                return 'DontPlay'
            else:  # Windy = Yes
                # Cek Temperature
                if row['Temperature'] == 'Hot':
                    return 'DontPlay'
                else:  # Temperature = Mild
                    return 'Play'

print("VALIDASI DENGAN SEMUA DATA (TREE YANG BENAR)")
print("=" * 80)
print("\nNo | Humidity | Outlook | Windy | Temp  | Actual    | Prediksi  | Status")
print("-" * 80)

correct = 0
for i, row in df.iterrows():
    pred = predict_manual_corrected(row)
    actual = row['Play']
    status = '✓' if pred == actual else '✗'
    if pred == actual:
        correct += 1
    print(f"{i+1:2} | {row['Humidity']:7} | {row['Outlook']:6} | {row['Windy']:4} | {row['Temperature']:5} | {actual:9} | {pred:9} | {status}")

accuracy = correct / len(df) * 100
print(f"\n{'='*80}")
print(f"AKURASI: {correct}/{len(df)} = {accuracy:.1f}%")
print(f"{'='*80}")

if accuracy == 100:
    print("\n✓ Tree sudah perfect dan sesuai dengan semua data!")
else:
    print("\n✗ Ada data yang tidak sesuai, perlu pengecekan ulang")

VALIDASI DENGAN SEMUA DATA (TREE YANG BENAR)

No | Humidity | Outlook | Windy | Temp  | Actual    | Prediksi  | Status
--------------------------------------------------------------------------------
 1 | High    | Sunny  | No   | Hot   | DontPlay  | DontPlay  | ✓
 2 | High    | Sunny  | Yes  | Hot   | DontPlay  | DontPlay  | ✓
 3 | High    | Cloudy | No   | Hot   | Play      | Play      | ✓
 4 | High    | Rainy  | No   | Mild  | Play      | Play      | ✓
 5 | Normal  | Rainy  | No   | Cold  | Play      | Play      | ✓
 6 | Normal  | Rainy  | Yes  | Cold  | Play      | Play      | ✓
 7 | Normal  | Cloudy | Yes  | Cold  | Play      | Play      | ✓
 8 | High    | Sunny  | No   | Mild  | DontPlay  | DontPlay  | ✓
 9 | Normal  | Sunny  | No   | Cold  | Play      | Play      | ✓
10 | Normal  | Sunny  | No   | Mild  | Play      | Play      | ✓
11 | Normal  | Sunny  | Yes  | Mild  | Play      | Play      | ✓
12 | High    | Sunny  | Yes  | Mild  | Play      | Play      | ✓
13 | Normal  | Sunny

## Verifikasi Rule Sets

In [ ]:
print("VERIFIKASI RULE SET")
print("=" * 80)

rules = [
    ("Rule 1: Humidity = Normal", df[df['Humidity'] == 'Normal']),
    ("Rule 2: Humidity=High & Outlook=Cloudy", df[(df['Humidity']=='High') & (df['Outlook']=='Cloudy')]),
    ("Rule 3: Humidity=High & Outlook=Rainy & Windy=No", df[(df['Humidity']=='High') & (df['Outlook']=='Rainy') & (df['Windy']=='No')]),
    ("Rule 4: Humidity=High & Outlook=Rainy & Windy=Yes", df[(df['Humidity']=='High') & (df['Outlook']=='Rainy') & (df['Windy']=='Yes')]),
    ("Rule 5: Humidity=High & Outlook=Sunny & Windy=No", df[(df['Humidity']=='High') & (df['Outlook']=='Sunny') & (df['Windy']=='No')]),
    ("Rule 6a: Humidity=High & Outlook=Sunny & Windy=Yes & Temp=Hot", df[(df['Humidity']=='High') & (df['Outlook']=='Sunny') & (df['Windy']=='Yes') & (df['Temperature']=='Hot')]),
    ("Rule 6b: Humidity=High & Outlook=Sunny & Windy=Yes & Temp=Mild", df[(df['Humidity']=='High') & (df['Outlook']=='Sunny') & (df['Windy']=='Yes') & (df['Temperature']=='Mild')])
]

for rule_name, data in rules:
    print(f"\n{rule_name}:")
    if len(data) > 0:
        print(data[['Humidity', 'Outlook', 'Windy', 'Temperature', 'Play']].to_string(index=False))
        predictions = data['Play'].values
        if len(set(predictions)) == 1:
            print(f"✓ Hasil: {predictions[0]} (konsisten)")
        else:
            print(f"✗ Hasil: {dict(Counter(predictions))} (TIDAK KONSISTEN!)")
    else:
        print("  Tidak ada data")

VERIFIKASI RULE SET

Rule 1: Humidity = Normal:
Humidity Outlook Windy Temperature Play
  Normal   Rainy    No        Cold Play
  Normal   Rainy   Yes        Cold Play
  Normal  Cloudy   Yes        Cold Play
  Normal   Sunny    No        Cold Play
  Normal   Sunny    No        Mild Play
  Normal   Sunny   Yes        Mild Play
  Normal   Sunny    No         Hot Play
  Normal  Cloudy    No         Hot Play
✓ Hasil: Play (konsisten)

Rule 2: Humidity=High & Outlook=Cloudy:
Humidity Outlook Windy Temperature Play
    High  Cloudy    No         Hot Play
    High  Cloudy   Yes        Mild Play
✓ Hasil: Play (konsisten)

Rule 3: Humidity=High & Outlook=Rainy & Windy=No:
Humidity Outlook Windy Temperature Play
    High   Rainy    No        Mild Play
✓ Hasil: Play (konsisten)

Rule 4: Humidity=High & Outlook=Rainy & Windy=Yes:
Humidity Outlook Windy Temperature     Play
    High   Rainy   Yes        Mild DontPlay
    High   Rainy   Yes        Mild DontPlay
✓ Hasil: DontPlay (konsisten)

Rule 5:

In [3]:
import graphviz
from collections import Counter
import pandas as pd # Added import for pandas
import math # Added import for math

# Install graphviz if not already installed
try:
    import graphviz
except ImportError:
    print("graphviz not found. Installing...")
    !pip install graphviz
    import graphviz

# --- Moved from cell cf78796f-4388-40b7-90aa-84e65a05a069 ---
def entropy(data):
    """Hitung entropy"""
    if len(data) == 0:
        return 0
    counts = Counter(data)
    total = len(data)
    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)
    return ent

def info_gain(df, attr, target):
    """Hitung information gain"""
    total_entropy = entropy(df[target])
    weighted_entropy = 0
    total = len(df)
    for value in df[attr].unique():
        subset = df[df[attr] == value]
        subset_entropy = entropy(subset[target])
        weight = len(subset) / total
        weighted_entropy += weight * subset_entropy
    return total_entropy - weighted_entropy

def split_info(df, attr):
    """Hitung split info"""
    total = len(df)
    split = 0
    for value in df[attr].unique():
        subset_size = len(df[df[attr] == value])
        if subset_size > 0:
            p = subset_size / total
            split -= p * math.log2(p)
    return split

def gain_ratio(df, attr, target):
    """Hitung gain ratio"""
    gain = info_gain(df, attr, target)
    split = split_info(df, attr)
    return gain / split if split != 0 else 0
# --- End of moved functions ---

class Node:
    def __init__(self, feature=None, value=None, results=None, children=None, is_leaf=False):
        self.feature = feature       # Feature to split on (for internal nodes)
        self.value = value           # Value that led to this node (from parent's perspective)
        self.results = results       # Class distribution (for leaf nodes) or majority class (for leaves/pruning)
        self.children = children if children is not None else {} # Dictionary mapping feature_value to child Node
        self.is_leaf = is_leaf       # True if this is a leaf node

    def __repr__(self):
        if self.is_leaf:
            return f"Leaf(Result={self.results})"
        else:
            return f"Node(Feature='{self.feature}', Children={len(self.children)})";

def build_decision_tree(df, attributes, target_attribute, parent_value=None):
    # If all target values are the same, it's a leaf node
    if len(df[target_attribute].unique()) == 1:
        return Node(value=parent_value, results=df[target_attribute].iloc[0], is_leaf=True)

    # If there are no more attributes to split on, or no data, return a leaf with the majority class
    if len(attributes) == 0 or len(df) == 0:
        return Node(value=parent_value, results=Counter(df[target_attribute]).most_common(1)[0][0], is_leaf=True)

    # Find the best attribute to split on using gain ratio
    best_gain_ratio = -1
    best_feature = None
    for attr in attributes:
        gr = gain_ratio(df, attr, target_attribute)
        if gr > best_gain_ratio:
            best_gain_ratio = gr
            best_feature = attr

    # If no gain ratio is found, or it's zero, make a leaf node with the majority class
    if best_feature is None or best_gain_ratio <= 0:
        return Node(value=parent_value, results=Counter(df[target_attribute]).most_common(1)[0][0], is_leaf=True)

    # Create the node for the best feature
    node = Node(feature=best_feature, value=parent_value)

    # Recursively build children nodes
    remaining_attributes = [attr for attr in attributes if attr != best_feature]
    for value in df[best_feature].unique():
        subset = df[df[best_feature] == value]
        # Skip empty subsets
        if len(subset) == 0:
            continue
        node.children[value] = build_decision_tree(subset, remaining_attributes, target_attribute, value)

    return node

# Visualize the tree (requires graphviz)
def visualize_tree(node, dot=None, parent_name=None, edge_label=None, node_id_counter=None):
    if node_id_counter is None:
        node_id_counter = [0]

    if dot is None:
        dot = graphviz.Digraph(comment='Decision Tree', format='png')
        dot.attr(rankdir='TB') # Top to bottom
        dot.attr('node', shape='box', style='filled', fontname='Helvetica', fontsize='12')
        dot.attr('edge', fontname='Helvetica', fontsize='10')

    current_node_id = f'node_{node_id_counter[0]}'
    node_id_counter[0] += 1

    if node.is_leaf:
        label = f"Result: {node.results}"
        dot.node(current_node_id, label, fillcolor='#d7e3fc') # Light blue for leaves
    else:
        label = f"Split on: {node.feature}"
        dot.node(current_node_id, label, fillcolor='#b0c4de') # Darker blue for internal nodes

    if parent_name:
        dot.edge(parent_name, current_node_id, label=edge_label)

    if not node.is_leaf:
        for branch_value, child_node in node.children.items():
            visualize_tree(child_node, dot, current_node_id, str(branch_value), node_id_counter)
    return dot


# Assuming df, target, and attributes are defined in previous cells
# Re-define df, target, and attributes to ensure they are available for this cell
df = pd.read_csv('decision.csv') # Re-load df
target = 'Play'
attributes = ['Outlook', 'Temperature', 'Humidity', 'Windy']

print("Building the Decision Tree using Gain Ratio...")
decision_tree_root = build_decision_tree(df, attributes, target)
print("Decision Tree built successfully.")

# Print simplified tree structure
print("\nTree Structure (simplified view):")
def print_tree(node, indent=0, branch_info="Root"):
    prefix = "  " * indent
    if node.is_leaf:
        print(f"{prefix}[{branch_info}] -> LEAF: {node.results}")
    else:
        print(f"{prefix}[{branch_info}] -> NODE: Split on {node.feature}")
        for value, child in node.children.items():
            print_tree(child, indent + 1, f"{node.feature}={value}")

print_tree(decision_tree_root)


print("\nVisualizing the Decision Tree (requires graphviz)...")
try:
    dot = visualize_tree(decision_tree_root)
    # Render the tree to a file and display it
    dot.render('decision_tree', view=True, cleanup=True)
    print("Decision tree visualization saved as 'decision_tree.png' and displayed.")
except Exception as e:
    print(f"Could not visualize the tree. Error: {e}. Make sure graphviz is installed and available in your PATH, or try running the cell with '!pip install graphviz' first if you see an import error.")
    print("You can still inspect the tree structure programmatically by exploring the 'decision_tree_root' object.")


# Add a prediction function to demonstrate the tree usage
def predict(tree_node, sample):
    if tree_node.is_leaf:
        return tree_node.results

    feature_value = sample[tree_node.feature]
    if feature_value in tree_node.children:
        return predict(tree_node.children[feature_value], sample)
    else:
        # Fallback: if an unseen value is encountered, predict the majority class of the current node's children
        # For this example, we'll assume all paths are covered by the training data.
        # A more robust solution might return the majority class of the data that reached this node.
        # For now, if no path matches, it implies an issue with unseen data or tree completeness for prediction.
        return None # Indicate prediction failure for unseen paths

print("\n--- Testing Prediction with the built tree ---")
# Pick a row from the original DataFrame to test
sample_data = df.iloc[0]
print(f"Sample data:\n{sample_data.to_dict()}")
predicted_class = predict(decision_tree_root, sample_data)
print(f"Predicted class: {predicted_class}")
print(f"Actual class: {sample_data['Play']}")

print("\n--- Validating all data with the built tree ---")
correct_predictions = 0
total_predictions = len(df)
for _, row in df.iterrows():
    predicted = predict(decision_tree_root, row)
    actual = row['Play']
    if predicted == actual:
        correct_predictions += 1

accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0
print(f"Accuracy on training data: {accuracy:.2f}% ({correct_predictions}/{total_predictions})")

Building the Decision Tree using Gain Ratio...
Decision Tree built successfully.

Tree Structure (simplified view):
[Root] -> NODE: Split on Humidity
  [Humidity=High] -> NODE: Split on Outlook
    [Outlook=Sunny] -> NODE: Split on Temperature
      [Temperature=Hot] -> LEAF: DontPlay
      [Temperature=Mild] -> NODE: Split on Windy
        [Windy=No] -> LEAF: DontPlay
        [Windy=Yes] -> LEAF: Play
    [Outlook=Cloudy] -> LEAF: Play
    [Outlook=Rainy] -> NODE: Split on Windy
      [Windy=No] -> LEAF: Play
      [Windy=Yes] -> LEAF: DontPlay
  [Humidity=Normal] -> LEAF: Play

Visualizing the Decision Tree (requires graphviz)...
Decision tree visualization saved as 'decision_tree.png' and displayed.

--- Testing Prediction with the built tree ---
Sample data:
{'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Windy': 'No', 'Play': 'DontPlay'}
Predicted class: DontPlay
Actual class: DontPlay

--- Validating all data with the built tree ---
Accuracy on training data: 100.